In [1]:
# Imports
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error



from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
import pandas as pd
import time


# Recherche du meilleur model de prévision
Environ 35 features, une target = débit, dataset total labelisé  environ 60 lignes

In [ ]:
# 1️⃣ Pipelines SANS réduction
def build_basic_pipelines():
    pipelines = {}

    pipelines["Ridge"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    pipelines["Lasso"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.01, max_iter=10000))
    ])

    pipelines["RandomForest"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=42))
    ])

    pipelines["SVR"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=1.0, epsilon=0.1))
    ])

    pipelines["KNN"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(
            n_neighbors=5,
            weights="distance"
        ))
    ])

    return pipelines

| Modèle       | Ce qu’il capte bien     |
| ------------ | ----------------------- |
| Ridge        | relations globales      |
| SVR          | relations non linéaires |
| KNN          | similarité locale       |
| RandomForest | interactions complexes  |


In [ ]:
# on crée un model de models à part pour pouvoir l'intégrer dans le pipeline PCA ensuite
from sklearn.ensemble import StackingRegressor    
stack_model = StackingRegressor(
    estimators=[
        ("ridge", Ridge()),
        ("svr", SVR(kernel="rbf", C=1.0, epsilon=0.1)),
        ("knn", KNeighborsRegressor(n_neighbors=3))
    ],
    final_estimator=Ridge()
    )

In [1]:
# 2️⃣ Pipelines AVEC PCA
def build_pca_pipelines(n_components=.97): # soit le nombre de composantes, soit la variance expliquée souhaitée (ex: 0.95 pour 95% de la variance)
    pipelines = {}

    '''Ce que fait le système qd on lance le pipeline :
    Entrainement :
        StandardScaler.fit_transform(X_train) (equivalent à scaler.fit(X_train) + scaler.transform(X_train))
        PCA.fit_transform(X_train)
        Model.fit(...)

    prediction/scoring : 
        StandardScaler.transform(X_test)
        PCA.transform(X_test)
        Model.predict(...)'''

    pipelines["Ridge_PCA"] = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components)),
        ("model", Ridge(alpha=1.0))
    ])

    pipelines["SVR_PCA"] = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components)),
        ("model", SVR(kernel="rbf", C=1.0, epsilon=0.1))
    ])
    
    pipelines["RandomForest"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=42))
    ])
    pipelines["KNN_PCA"] = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components)),
        ("model", KNeighborsRegressor(
            n_neighbors=5,
            weights="distance"
        ))
    ])


    pipelines["Stacked_PCA"] = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components)),
        ("model", stack_model)
    ])

    return pipelines

In [ ]:
# 3️⃣ Pipelines avec SelectKBest (souvent meilleur que PCA)
def build_selectkbest_pipelines(k=15):
    pipelines = {}

    pipelines["Ridge_SelectK"] = Pipeline([
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=f_regression, k=k)),
        ("model", Ridge(alpha=1.0))
    ])

    pipelines["SVR_SelectK"] = Pipeline([
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=f_regression, k=k)),
        ("model", SVR(kernel="rbf", C=1.0, epsilon=0.1))
    ])

    pipelines["KNN_SelectK"] = Pipeline([
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=f_regression, k=k)),
        ("model", KNeighborsRegressor(
            n_neighbors=5,
            weights="distance"
        ))
    ])

    return pipelines

In [ ]:
# 4️⃣ Fonction pour tout construire
def build_all_pipelines():
    pipelines = {}

    pipelines.update(build_basic_pipelines())
    pipelines.update(build_pca_pipelines(n_components=15))
    pipelines.update(build_selectkbest_pipelines(k=15))

    return pipelines

# Analyses avec cross-validations


Pour chaque modèle, calcul :

✅ MAE moyen en CV (robuste)
✅ Variabilité (std)
✅ R² sur test réel
✅ Temps exécution
✅ Nombre de vecteurs principaux (si PCA)


# 🔒 Zéro data leakage

Avec 50–100 samples :

    Le R² test peut beaucoup varier

    Le MAE CV est souvent plus stable

Donc pour choisir le modèle :

👉 Priorité au CV_MAE_mean
👉 Test_R2 pour validation finale

In [ ]:
def benchmark_models(X, y):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    pipelines = build_all_pipelines()
    results = []

    for name, pipe in pipelines.items():

        # Cross-validation
        cv_scores = cross_val_score(
            pipe,
            X_train,
            y_train,
            cv=5,
            scoring="neg_mean_absolute_error"
        )

        mae_mean = -np.mean(cv_scores)
        mae_std = np.std(cv_scores)

        
        pipe.fit(X_train, y_train)
        

        # Mesure du temps du run du pipeline de A à Z (data prep + .predict + score  = metric(y_test, y_pred))
        start_time = time.time()
        # Score test
        r2_test = pipe.score(X_test, y_test)
        test_time = time.time() - start_time

        # Nombre de composantes PCA (si présent)
        n_pca_components = None
        if "pca" in pipe.named_steps:
            n_pca_components = pipe.named_steps["pca"].n_components_

        results.append({
            "Model": name,
            "CV_MAE_mean": mae_mean,
            "CV_MAE_std": mae_std,
            "Test_R2": r2_test,
            "Test_time_sec": test_time,
            "PCA_components": n_pca_components
        })

    results_df = (
        pd.DataFrame(results)
        .sort_values("CV_MAE_mean")
        .reset_index(drop=True)
    )

    return results_df

# Entrainement sur tout le jeu de calibration (sans train/test)
Seconde piste d'analyse: Si peu de valeurs dans tout le set (moins de 100): pas forcément de découpe train/test mais cross_val_score: 
Parce que la cross-validation :

    découpe elle-même en 5 folds

    entraîne sur 4

    valide sur 1

    répète 5 fois

In [ ]:
def benchmark_models_RepeatedKFold(X, y):
    
    pipelines = build_all_pipelines()
    results = {}

    rkf = RepeatedKFold(
        n_splits=5,
        n_repeats=10,
        random_state=42
    )

    for name, model in pipelines.items():
        
        scores = cross_val_score(
            model,
            X,
            y,
            cv=rkf,
            scoring="neg_mean_absolute_error"
        )

        mae_scores = -scores
        
        results[name] = {
            "MAE_mean": np.mean(mae_scores),
            "MAE_std": np.std(mae_scores)
        }

    return results

# Code principal
Lancement du bench sur la base du fichier csv des données de calibration

In [ ]:
# ==========================================
# 🔹 CODE PRINCIPAL
# ==========================================

import pandas as pd

# 1️⃣ Charger le dataset
calib = pd.read_csv("calibration_features.csv", sep=",")

# 2️⃣ Séparer features / target
X = calib.iloc[:, :-1]
y = calib.iloc[:, -1]

print("Shape dataset :", calib.shape)
print("Nb features :", X.shape[1])
print("Nb samples :", X.shape[0])
print("Target :", calib.columns[-1])
print("-" * 50)


# ==========================================
# 🔹 BENCHMARK CLASSIQUE (train/test + CV)
# ==========================================

print("🔎 Benchmark avec split train/test + CV")
results = benchmark_models(X, y)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("CV_MAE_mean")

print("\nRésultats triés par CV_MAE_mean :")
print(results_df)


# ==========================================
# 🔹 BENCHMARK FULL CV (petits datasets)
# ==========================================

print("\n🔎 Benchmark full Cross-Validation (5 folds)")
results_cv = benchmark_models_RepeatedKFold(X, y)

results_cv_df = pd.DataFrame(results_cv).T
results_cv_df = results_cv_df.sort_values("MAE_mean")

print("\nRésultats triés par MAE_mean :")
print(results_cv_df)


In [ ]:
# ==========================================
# 🔹 CODE PRINCIPAL N°2 avec randomSearch des hyperparm de modèles
# ==========================================

# faire 3 jeux de tests avec des calibrations différentes : 16, 24, 32 Khz. Les télé^phones sot en 16kHz ou variabl e(sr = None)

# from sklearn.model_selection import GridSearchCV
# import pandas as pd
# import numpy as np

# 🔹 Charger le dataset
df = pd.read_csv("calibration_features.csv", sep=",")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# 🔹 Pipelines déjà définis : build_all_pipelines()
pipelines = build_all_pipelines()

# 🔹 Param grids adaptés pour petit dataset
param_grid = {
    "KNN": {
        "model__n_neighbors": [3, 5, 7],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    },
    "Ridge": {
        "model__alpha": [0.01, 0.1, 1.0, 10.0],
        "model__solver": ["auto", "svd"]
    },
    "Lasso": {
        "model__alpha": [0.001, 0.01, 0.1, 1.0],
        "model__max_iter": [5000, 10000],
        "model__selection": ["cyclic", "random"]
    },
    "RandomForest": {
        "model__n_estimators": [50, 100],
        "model__max_depth": [None, 3, 5, 10],
        "model__min_samples_split": [2, 4, 6],
        "model__min_samples_leaf": [1, 2, 3],
        "model__max_features": ["auto", "sqrt"]
    },
    "SVR": {
        "model__C": [0.1, 1.0, 10.0],
        "model__epsilon": [0.01, 0.1, 0.5],
        "model__kernel": ["rbf", "linear"],
        "model__gamma": ["scale", "auto"]
    }
}

# 🔹 Stocker les résultats
results = []

for name, pipeline in pipelines.items():
    print(f"\n===== GridSearchCV pour {name} =====")
    
    if name not in param_grid:
        print(f"Aucun param_grid défini pour {name}, skip.")
        continue
    
    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid[name],
        scoring="neg_mean_absolute_error",
        cv=5,
        n_jobs=-1
    )
    
    gs.fit(X, y)
    
    print("Meilleurs paramètres :", gs.best_params_)
    print("Meilleur MAE CV :", -gs.best_score_)
    
    results.append({
        "Model": name,
        "Best_Params": gs.best_params_,
        "Best_CV_MAE": -gs.best_score_
        "Best_Estimator": rs.best_estimator_
    })

# 🔹 Résumé dans un DataFrame
results_df = pd.DataFrame(results).sort_values("Best_CV_MAE")
print("\n===== Résumé final trié par MAE =====")
print(results_df)
